# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Artemisdevv/Flyrank-internship-ml-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


My lane is **scoring / ranking**. The goal is to give each content page a score so that human reviewers can check the most important pages first.

For this exercise, we can use classification to predict whether a page has a decline label, then use the predicted score to rank the pages.

In [7]:
# @title
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I will use `is_declining_label` as the target. It is `1` when `trend_direction` is `"down"` and `0` otherwise.

This is based on the current data, so it does not predict future decline. I will also avoid using `trend_direction` and `trend_pct` as model inputs because they are used to create the target.

In [8]:
# @title
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df = pd.read_csv(
    "https://raw.githubusercontent.com/Artemisdevv/Flyrank-internship-ml-starter/main/data/raw/content_refresh_anonymized.csv"
)
df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].drop_duplicates(subset="content_id").copy()

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df.shape)
print(df["is_declining_label"].value_counts())

(30000, 45)
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My main metric is **Precision@50**. It measures how many of the 50 highest-ranked pages have the positive decline label.

A higher score means more relevant pages appear at the top of the review queue. This is useful because reviewers have limited time. It should be measured on clients that were not used for training.

In [9]:
# @title
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Precision@50 will be calculated after the model produces ranking scores.
def precision_at_50(ranked_pages):
    return ranked_pages.head(50)["is_declining_label"].mean()

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row represents one content page.** Each row contains page information and search or engagement measurements collected over 90 days.

`content_id` identifies the page, while `client_id` identifies its client. These are identifiers and should not be used as model features.

The dataframe below shows the actual data used for this task.

In [10]:
# @title
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
display(df[[
    "content_id",
    "client_id",
    "impressions_90d",
    "content_age_days",
    "trend_direction",
    "is_declining_label"
]].head())

print("Number of pages:", len(df))
print("Number of clients:", df["client_id"].nunique())

,content_id,client_id,impressions_90d,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,263,down,1


Number of pages: 30000
Number of clients: 32


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule might miss patterns between different signals like impressions, clicks and page age.

ML can look at these signals together and help rank pages by priority. We can then compare it with simple rules to see which works better.

The final decision is still made by a human reviewer.

In [11]:
# @title
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Some possible input features
features = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

print("Available features:")
print([col for col in features if col in df.columns])

print("\nColumns to exclude:")
print(["trend_direction", "trend_pct", "content_id", "client_id"])

Available features:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 'content_age_days']

Columns to exclude:
['trend_direction', 'trend_pct', 'content_id', 'client_id']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.